#**2nd Week**

##**Задачи - JOIN (Продвинутый уровень)**

В этом тесте вам предстоит решить практические задачи продвинутого уровня на тему "JOIN".

Выведите список оценок и их бронирований, даже если оценка не была оставлена для бронирования. Отобразите ID оценки, саму оценку, ID бронирования, дату заезда, дату выезда, ID клиента, номер комнаты. Выведите информацию только для бронирований, где имя клиента "Георгий", и цена за номер больше 20000, отсортируйте по ID бронирования и выведите первые 30 записей.

In [ ]:
SELECT 
    r.rating_id,
    r.rating_value,
    b.booking_id,
    b.check_in_date,
    b.check_out_date,
    b.renter_id,
    b.room_number
FROM 
    bookings b
LEFT JOIN 
    ratings r ON b.booking_id = r.booking_id
JOIN 
    clients c ON b.renter_id = c.id
JOIN 
    rooms r2 ON b.room_number = r2.room_number
WHERE 
    c.first_name = 'Георгий'
    AND r2.price_per_night > 20000
ORDER BY 
    b.booking_id
LIMIT 30;


Выведите список клиентов (имя и фамилию) и их оценки, даже если клиент не оставил оценку. Отобразите только оценки для бронирований, которые были оплачены 'наличными', номер был категории "Президентский" и телефон клиента начинается с "+7". Дополнительно выведите ID клиента и ID бронирования. Сортируйте по ID клиента, потом по ID бронирования. Выведите только первые 40 записей.

In [ ]:
SELECT 
    c.first_name,
    c.last_name,
    r.rating_value,
    b.renter_id AS id,
    b.booking_id
FROM 
    clients c
LEFT JOIN 
    bookings b ON c.id = b.renter_id
LEFT JOIN 
    ratings r ON b.booking_id = r.booking_id
JOIN 
    rooms r2 ON b.room_number = r2.room_number
JOIN 
    payments p ON b.booking_id = p.booking_id
WHERE 
    p.payment_method = 'наличными'
    AND r2.type_name = 'Президентский'
    AND c.phone_number LIKE '+7%'
ORDER BY 
    c.id, b.booking_id
LIMIT 40;


Выведите топ самых хорошо оцененных типов номеров, которые были оплачены банковской картой, включая тип номера, общее количество бронирований (total_bookings), среднюю оценку для этого типа номера (average_rating). Добавьте колонку booking_rank, в которой будет рейтинг типа номера по количеству бронирований.

In [ ]:
WITH room_ratings AS (
    SELECT 
        r.type_name as type_name,
        COUNT(DISTINCT b.booking_id) AS total_bookings, -- Count distinct bookings
        AVG(r.rating_value) AS average_rating
    FROM 
        rooms r
    JOIN 
        bookings b ON r.room_number = b.room_number
    LEFT JOIN 
        ratings r ON b.booking_id = r.booking_id
    JOIN 
        payments p ON b.booking_id = p.booking_id
    WHERE 
        p.payment_method = 'картой'
    GROUP BY 
        r.type_name
)
SELECT 
    type_name,
    total_bookings,
    average_rating,
    RANK() OVER (ORDER BY total_bookings DESC) AS booking_rank
FROM 
    room_ratings
ORDER BY 
    booking_rank;


Найдите для каждого клиента максимальную сумму, потраченную на бронирование ,а также количество оплаченных бронирований. Выведите ID клиента, его имя, фамилию, максимальную сумму бронирования (max_payment) и количество оплаченных бронирований (count_paid_bookings). Если клиент не совершал бронирования, то оба этих поля должны быть 0. Отсортируйте результат по убыванию максимальной суммы бронирования и по id клиента, отфильтруйте результат по имени 'Радислав' и ограничьте вывод 50 записями.

In [ ]:
SELECT 
    c.id,
    c.first_name,
    c.last_name,
    COALESCE(MAX(p.amount_paid), 0) AS max_payment,
    COALESCE(COUNT(DISTINCT b.booking_id), 0) AS count_paid_bookings
FROM 
    clients c
LEFT JOIN 
    bookings b ON c.id = b.renter_id
LEFT JOIN 
    payments p ON b.booking_id = p.booking_id
WHERE 
    c.first_name = 'Радислав'
    
GROUP BY 
    c.id, c.first_name, c.last_name
ORDER BY 
    max_payment DESC, c.id
LIMIT 50;


Выведите список клиентов с их именами, фамилиями и средним количеством дней, проведенных в отеле за все их бронирования (average_days). Отсортируйте результаты по убыванию среднего количества дней. Выведите только первые 10 записей.
 
Примечание:  Используем функцию JULIANDAY для получения количества дней между датами. Пример: JULIANDAY(date_2) -JULIANDAY(date_1)

In [ ]:
SELECT 
    c.first_name,
    c.last_name,
    AVG(JULIANDAY(b.check_out_date) - JULIANDAY(b.check_in_date)) AS average_days
FROM 
    clients c
JOIN 
    bookings b ON c.id = b.renter_id
GROUP BY 
    c.id, c.first_name, c.last_name
ORDER BY 
    average_days DESC
LIMIT 10;


Выведите информацию о бронированиях клиентов, которые бронировали номера с мини-баром. Для каждого бронирования выведите ID бронирования, даты заезда и выезда, номер комнаты, имя и фамилию клиента и среднюю оценку этого клиента по всем его бронированиям (average_rating). Отсортируйте по ID бронирования и ограничьте вывод 50 записями.

Подсказка: не забывайте, что фильтрация данных происходит до выполнения оконок.

In [ ]:
SELECT
    b.booking_id,
    b.check_in_date,
    b.check_out_date,
    b.room_number,
    c.first_name,
    c.last_name,
    (SELECT AVG(rating_value) 
     FROM ratings r 
     WHERE r.booking_id = b.booking_id) AS average_rating
FROM
    bookings b
JOIN
    rooms ro ON b.room_number = ro.room_number
JOIN
    clients c ON b.renter_id = c.id
WHERE
    ro.has_minibar = 1
ORDER BY
    b.booking_id
LIMIT 50;


Выведите список всех комнат. Для каждой комнаты выведите процент бронирований, оплаченных 'картой' (bank_card_count), процент бронирований, оплаченных 'СБП' (sbp_count), и процент бронирований, оплаченных 'наличными' (cash_count). Округлите все проценты до двух знаков после запятой. Комнаты, по которым не было бронирований, учитывать не надо.

In [ ]:
SELECT 
    r.room_number,
    r.room_type,
    ROUND(COALESCE(SUM(CASE WHEN p.payment_method = 'карта' THEN 1 ELSE 0 END), 0) * 100.0 / COUNT(b.booking_id), 2) AS bank_card_count,
    ROUND(COALESCE(SUM(CASE WHEN p.payment_method = 'СБП' THEN 1 ELSE 0 END), 0) * 100.0 / COUNT(b.booking_id), 2) AS sbp_count,
    ROUND(COALESCE(SUM(CASE WHEN p.payment_method = 'наличными' THEN 1 ELSE 0 END), 0) * 100.0 / COUNT(b.booking_id), 2) AS cash_count
FROM 
    rooms r
JOIN 
    bookings b ON r.room_number = b.room_number
JOIN 
    payments p ON b.booking_id = p.booking_id
GROUP BY 
    r.room_number, r.room_type
HAVING 
    COUNT(b.booking_id) > 0;  -- Exclude rooms with no bookings

Найдите самый популярный тип номера среди клиентов, проживающих в городе "Ростов", укажите этот тип номера и количество бронирований для этого типа номера (total_bookings).
Примечание: Ищем клиентов, у которых в адресе присутствует слово "Ростов"

In [ ]:
SELECT
    ro.type_name,
    COUNT(b.booking_id) AS total_bookings
FROM
    clients c
JOIN
    bookings b ON c.id = b.renter_id
JOIN
    rooms ro ON b.room_number = ro.room_number
WHERE
    c.address LIKE '%Ростов%'  -- Проверка на наличие слова "Ростов" в адресе
GROUP BY
    ro.type_name
ORDER BY
    total_bookings DESC
LIMIT 1;
